# Population Exposure Model

This notebook builds a population exposure model for Night and Day scenarios.

## Required input files

Place these files in the same folder as the notebook:

| File | Purpose | Required columns / notes |
|---|---|---|
| `worldpop.tif` | WorldPop population raster | GeoTIFF population raster |
| `buildings.shp` | Building footprint shapefile | Must include `ID` column |
| `osmbuildings.shp` | Building type shapefile 
| `A1_CORESIDENCE_SUBNATIONAL_DATASET.csv` | Household-size probabilities 

The study-area bounding box is fixed for `Marzameni`.

In [ ]:
import sys
print(sys.executable)
!conda install -y -c conda-forge geopandas rasterio shapely pyproj fiona

In [ ]:
# ============================================================
# Cell 1 — Import Libraries
# ============================================================

#!pip install geopandas rasterio osmnx matplotlib

import os
import random
import pandas as pd
import numpy as np
import geopandas as gpd
import rasterio
import matplotlib.pyplot as plt

from rasterio.windows import from_bounds
from shapely.geometry import box

os.makedirs("outputs", exist_ok=True)


In [ ]:
# ============================================================
# Study-Area BBox
# ============================================================
# In this example, the bounding box is for Marzameni, Sicily, Italy.
# The same workflow can be applied to any other area by replacing these four coordinates with another bounding box.

bbox = (
    15.087448772194595,  # xmin
    36.72217897074608,   # ymin
    15.129462687805406,  # xmax
    36.76231628625393    # ymax
)

print("Study-area bbox:", bbox)


In [ ]:
# ============================================================
# Load Building Shapefile
# ============================================================
# Read the building footprint shapefile and convert it to WGS84 coordinates (EPSG:4326)
# In this example, the building layer was extracted from the "Presidenza del Consiglio dei Ministri - Dipartimento della Protezione Civile" dataset.
# Source: https://github.com/pcm-dpc/DPC-Aggregati-Strutturali-ITG-Isole
# However, any building footprint dataset can be used as long as:
#   1. Each building has a unique identifier column
#   2. The dataset contains valid polygon geometries
# If using another dataset, simply replace "IDAG" below with the corresponding unique building ID column name.

building_file = "Marzameni.shp"

buildings = gpd.read_file(building_file)
buildings = buildings.to_crs("EPSG:4326")

xmin, ymin, xmax, ymax = bbox
bbox_polygon = box(xmin, ymin, xmax, ymax)

buildings_bbox = buildings[buildings.intersects(bbox_polygon)].copy()

buildings_bbox = buildings_bbox[["IDAG", "geometry"]].copy()
buildings_bbox = buildings_bbox.rename(columns={"IDAG": "BuildingID"})
buildings_bbox = buildings_bbox.reset_index(drop=True)

print("Buildings inside bbox:", len(buildings_bbox))

buildings_bbox.head()


In [ ]:
# ============================================================
# Read Prepared OSM Building-Type File and Merge
# ============================================================
# Read the OpenStreetMap (OSM) building layer and convert it to WGS84 coordinates.
# The OSM dataset used here was downloaded from Geofabrik.
# Source: https://download.geofabrik.de/
# In this workflow, OSM buildings are used to select the type of buildings (e.g., residential, commercial, industrial).
# It is also possible to retrieve OSM data directly using APIs such as:
#   - Overpass API
#   - OSMnx
# however Large areas may exceed API limits, therefore regional downloads (e.g., from Geofabrik) are usually preferred for large-scale studies.

# ============================================================
# Read Prepared OSM Building-Type File and Merge
# ============================================================

osm_file = "OsmMarzameni.shp"

osm = gpd.read_file(osm_file)
osm = osm.to_crs("EPSG:4326")

# Use "type" column from your shapefile
osm = osm[["type", "geometry"]].copy()

def building_type(x):
    x = str(x).strip().lower()

    industrial_classes = [
        "construction",
        "greenhouse",
        "farm",
        "industrial"
    ]

    commercial_classes = [
        "church",
        "school",
        "service",
        "kiosk",
        "train_station",
        "shed",
        "hotel",
        "farm_auxiliary",
        "bunker",
        "supermarket"
    ]

    if x in industrial_classes:
        return "industrial"
    elif x in commercial_classes:
        return "commercial"
    else:
        return "residential"


osm["BuildingType"] = osm["type"].apply(building_type)

# Remove old classification columns from buildings_bbox if they exist
for col in ["BuildingType", "BuildingType_left", "BuildingType_right", "type", "type_left", "type_right"]:
    if col in buildings_bbox.columns:
        buildings_bbox = buildings_bbox.drop(columns=[col])

# Ensure CRS match
buildings_bbox = buildings_bbox.to_crs("EPSG:4326")

# Calculate centroids safely using projected CRS
buildings_centroid = buildings_bbox.to_crs("EPSG:32633").copy()
buildings_centroid["geometry"] = buildings_centroid.geometry.centroid
buildings_centroid = buildings_centroid.to_crs("EPSG:4326")

# Spatial join
joined = gpd.sjoin(
    buildings_centroid,
    osm[["type", "BuildingType", "geometry"]],
    how="left",
    predicate="intersects"
)

# Keep first match if duplicate matches happen
joined = joined[~joined.index.duplicated(keep="first")]

# Check join output columns
print("Joined columns:")
print(list(joined.columns))

# Find correct BuildingType column after join
if "BuildingType" in joined.columns:
    bt_col = "BuildingType"
elif "BuildingType_right" in joined.columns:
    bt_col = "BuildingType_right"
else:
    raise KeyError(f"No BuildingType column found after join. Columns are: {list(joined.columns)}")

if "type" in joined.columns:
    type_col = "type"
elif "type_right" in joined.columns:
    type_col = "type_right"
else:
    type_col = None

# Add results back to buildings_bbox
buildings_bbox["BuildingType"] = joined[bt_col].reindex(buildings_bbox.index)

if type_col is not None:
    buildings_bbox["OSM_type"] = joined[type_col].reindex(buildings_bbox.index)
else:
    buildings_bbox["OSM_type"] = None

# Buildings with no OSM match become residential
buildings_bbox["BuildingType"] = buildings_bbox["BuildingType"].fillna("residential")
buildings_bbox["OSM_type"] = buildings_bbox["OSM_type"].fillna("no_osm_match")

# Final subsets
residential_buildings = buildings_bbox[
    buildings_bbox["BuildingType"] == "residential"
].copy()

commercial_buildings = buildings_bbox[
    buildings_bbox["BuildingType"] == "commercial"
].copy()

industrial_buildings = buildings_bbox[
    buildings_bbox["BuildingType"] == "industrial"
].copy()

# Report only
print("\nFinal Building Classification")
print("-----------------------------------")
print("Residential :", len(residential_buildings))
print("Commercial  :", len(commercial_buildings))
print("Industrial  :", len(industrial_buildings))
print("Total       :", len(buildings_bbox))



In [ ]:
worldpop_file = "ita_pop_2025_CN_1km_R2025A_UA_v1.tif"

with rasterio.open(worldpop_file) as src:

    # Extract raster window using study-area bbox
    window = from_bounds(*bbox, transform=src.transform)

    # Read raster data
    pop_data = src.read(1, window=window)

    # Window transform
    window_transform = src.window_transform(window)

# ============================================================
# Clean raster
# ============================================================

pop_data = np.where(pop_data < 0, 0, pop_data)
pop_data = np.nan_to_num(pop_data)

# ============================================================
# Total population
# ============================================================

target_population = int(round(pop_data.sum()))

print("Target population from WorldPop:", target_population)

# ============================================================
# Print population in each raster cell
# ============================================================

rows, cols = pop_data.shape

print("\nPopulation in each raster cell:\n")

cell_id = 1

for r in range(rows):
    for c in range(cols):

        value = pop_data[r, c]

        if value > 0:

            print(f"Cell {cell_id}: {value:.2f}")

            cell_id += 1

# ============================================================
# Raster extent for plotting
# ============================================================

left = window_transform.c
top = window_transform.f

right = left + pop_data.shape[1] * window_transform.a
bottom = top + pop_data.shape[0] * window_transform.e

extent = [left, right, bottom, top]

# ============================================================
# Plot raster + buildings
# ============================================================

fig, ax = plt.subplots(figsize=(12, 12))

# Plot raster
img = ax.imshow(
    pop_data,
    extent=extent,
    cmap="viridis",
    origin="upper"
)

# Plot building footprints
buildings_bbox.boundary.plot(
    ax=ax,
    color="white",
    linewidth=0.3
)

# ============================================================
# Final formatting
# ============================================================

cbar = plt.colorbar(img, ax=ax)
cbar.set_label("Population")

ax.set_title("WorldPop Population Raster with Buildings")

plt.show()

In [ ]:
# ============================================================
# Household Size Probabilities from CORESIDENCE
# ============================================================
# Read the CORESIDENCE dataset.This dataset provides statistical distributions of household sizes for different countries, regions, and years.
# In this example:
#   Country  = Italy (ITA)
#   Region   = Islands
#   Year     = 2020
# The extracted probabilities are used to simulate realistic household sizes during population allocation.
# Source: https://github.com/JuanGaleano/CORESIDENCE

coresidence_file = "A1_CORESIDENCE_SUBNATIONAL_DATASET.csv"

core = pd.read_csv(coresidence_file)

# Select Italy - Islands - 2019
row = core[
    (core["C1"] == "ITA") &
    (core["C6"] == "Islands") &
    (core["T1"] == 2019)
].iloc[0]

# Household sizes
family_sizes = np.array([1, 2, 3, 4, 5, 6])

# Probabilities
family_probs = row[
    ["HS01", "HS02", "HS03", "HS04", "HS05", "HS06"]
].astype(float).values

# Normalize
family_probs = family_probs / family_probs.sum()

# ============================================================
# Print probabilities
# ============================================================

print("Household Size Probabilities\n")

for size, prob in zip(family_sizes, family_probs):
    print(f"{size} person household: {prob:.3f}")

# ============================================================
# Compute CDF
# ============================================================

cdf = np.cumsum(family_probs)

# ============================================================
# Horizontal-step CDF (without vertical connectors)
# ============================================================

plt.figure(figsize=(8, 5))

for i in range(len(family_sizes) - 1):

    plt.hlines(
        y=cdf[i],
        xmin=family_sizes[i],
        xmax=family_sizes[i + 1],
        linewidth=2
    )

# Last horizontal segment
plt.hlines(
    y=cdf[-1],
    xmin=family_sizes[-1],
    xmax=family_sizes[-1] + 1,
    linewidth=2
)

# Points
plt.scatter(
    family_sizes,
    cdf
)

plt.xticks(family_sizes)

plt.ylim(0, 1.05)

plt.xlabel("Household Size")
plt.ylabel("Cumulative Probability")

plt.title("Horizontal-Step CDF of Household Size Distribution")

plt.grid(True)

plt.show()

In [ ]:
# ============================================================
# Allocate Population Cell by Cell
# ============================================================

# For each residential building, use its centroid to identify which WorldPop raster cell it belongs to.

residential_buildings = residential_buildings.copy()
residential_buildings["centroid"] = residential_buildings.geometry.centroid

coords = [(p.x, p.y) for p in residential_buildings["centroid"]]

with rasterio.open("ita_pop_2025_CN_1km_R2025A_UA_v1.tif") as src:
    rows_cols = [src.index(x, y) for x, y in coords]
    pop_values = [v[0] for v in src.sample(coords)]

residential_buildings["RasterRow"] = [r for r, c in rows_cols]
residential_buildings["RasterCol"] = [c for r, c in rows_cols]
residential_buildings["CellPopulation"] = [
    max(0, int(round(v))) for v in pop_values
]
residential_buildings = residential_buildings[
    residential_buildings["CellPopulation"] > 0
].copy()

results = []

# Now allocate population separately inside each WorldPop raster cell
for (row, col), group in residential_buildings.groupby(["RasterRow", "RasterCol"]):
    # Population target of this specific WorldPop cell
    cell_population = int(group["CellPopulation"].iloc[0])
    # Residential buildings located inside this cell
    building_ids = group["BuildingID"].tolist()
    random.shuffle(building_ids)

    assigned_people = 0

    for bid in building_ids:

        if assigned_people >= cell_population:
            size = 0
        else:
            # Randomly sample household size
            size = np.random.choice(family_sizes, p=family_probs)

            # Do not exceed this cell population
            if assigned_people + size > cell_population:
                size = cell_population - assigned_people

        assigned_people += size

        results.append({
            "BuildingID": bid,
            "RasterRow": row,
            "RasterCol": col,
            "CellPopulation": cell_population,
            "HouseholdSize": size
        })

buildings_population = pd.DataFrame(results)

print("Number of populated buildings:", (buildings_population["HouseholdSize"] > 0).sum())

buildings_population.head()

buildings_population.to_csv(
    "outputs/building_population_assignment.csv",
    index=False
)
print("Saved: outputs/building_population_assignment.csv")


In [ ]:
# ============================================================
# Plot Population Distribution
# ============================================================

plot_df = residential_buildings.merge(
    buildings_population,
    on="BuildingID",
    suffixes=("", "_pop")
)

plot_df = plot_df[plot_df["HouseholdSize"] > 0].copy()

# Check geometry columns
geom_cols = [
    col for col in plot_df.columns
    if str(plot_df[col].dtype) == "geometry"
]

print("Geometry columns before fix:", geom_cols)

# Keep only the main geometry column
main_geom = residential_buildings.geometry.name

if main_geom not in plot_df.columns:
    main_geom = geom_cols[0]

# Convert extra geometry columns to WKT text, or drop them
for col in geom_cols:
    if col != main_geom:
        plot_df[col + "_wkt"] = plot_df[col].to_wkt()
        plot_df = plot_df.drop(columns=[col])

# Rebuild as clean GeoDataFrame
plot_df = gpd.GeoDataFrame(
    plot_df,
    geometry=main_geom,
    crs=residential_buildings.crs
)

# Rename active geometry to standard "geometry"
if main_geom != "geometry":
    plot_df = plot_df.rename_geometry("geometry")

print("Geometry columns after fix:", [
    col for col in plot_df.columns
    if str(plot_df[col].dtype) == "geometry"
])

# Plot
fig, ax = plt.subplots(figsize=(10, 10))

plot_df.plot(
    column="HouseholdSize",
    ax=ax,
    legend=True
)

ax.set_title("Night-Time Population Distribution")
ax.set_axis_off()

plt.show()

# Save
os.makedirs("outputs", exist_ok=True)

plot_df.to_file(
    "outputs/night_time_population_buildings.gpkg",
    driver="GPKG"
)

print("Saved: outputs/night_time_population_buildings.gpkg")

In [ ]:
# ============================================================
# The daytime scenario
# ============================================================

# The daytime scenario is derived from the nighttime population.
# First, expand the building-level nighttime population into individuals.
# Each individual receives:
#   - PersonID
#   - Home BuildingID
#   - Night-time location

people = []

person_id = 0

for _, row in buildings_population.iterrows():

    building_id = row["BuildingID"]
    household_size = int(row["HouseholdSize"])

    for i in range(household_size):

        people.append({
            "PersonID": person_id,
            "HomeBuildingID": building_id,
            "NightBuildingID": building_id
        })

        person_id += 1

people_df = pd.DataFrame(people)

print("Night-time individuals:", len(people_df))
people_df.head()

In [ ]:
# Define Daytime Population Groups
# The daytime scenario is based on four groups derived from ISTAT daily mobility data for Sicily.
# Groups:
#   Static individuals     = 46%
#   Internal movers        = 25%
#   Incoming commuters     = 21%
#   Outgoing commuters     = 8%
# Outgoing commuters are excluded from the daytime scenario because they leave the city during the day.

day_roles = [
    "static",
    "internal_mover",
    "incoming_commuter",
    "outgoing_commuter"
]
day_role_probs = [
    0.46,
    0.25,
    0.21,
    0.08
]

people_df["DayRole"] = np.random.choice(
    day_roles,
    size=len(people_df),
    p=day_role_probs
)

print(people_df["DayRole"].value_counts(normalize=True))

In [ ]:
# Define Daytime Location Probabilities

# Each daytime group is assigned to a daytime environment.
# Static individuals:
#   75–85% residential buildings
#   15–25% open spaces
# Internal movers:
#   25–35% non-residential buildings
#   65–75% open spaces
# Incoming commuters:
#   15–25% non-residential buildings
#   75–85% open spaces
# Outgoing commuters:
#   absent from the daytime scenario

# ============================================================
# Define Daytime Location Probabilities
# ============================================================

def assign_day_location(role):

    if role == "static":
        p_home = np.random.uniform(0.75, 0.85)

        return np.random.choice(
            ["home", "open_space"],
            p=[p_home, 1 - p_home]
        )

    elif role == "internal_mover":
        p_nonres = np.random.uniform(0.25, 0.35)

        return np.random.choice(
            ["non_residential", "open_space"],
            p=[p_nonres, 1 - p_nonres]
        )

    elif role == "incoming_commuter":
        p_nonres = np.random.uniform(0.15, 0.25)

        return np.random.choice(
            ["non_residential", "open_space"],
            p=[p_nonres, 1 - p_nonres]
        )

    else:
        return "absent"


people_df["DayLocationType"] = people_df["DayRole"].apply(assign_day_location)

# ============================================================
# Report numbers
# ============================================================

location_counts = people_df["DayLocationType"].value_counts()
location_percent = people_df["DayLocationType"].value_counts(normalize=True) * 100

print("Daytime location sample numbers")
print("--------------------------------")

for loc in location_counts.index:
    print(f"{loc}: {location_counts[loc]} people ({location_percent[loc]:.2f}%)")

print("\nTotal people:", len(people_df))

# Optional cross-tab by role
print("\nDayRole × DayLocationType:")
print(pd.crosstab(people_df["DayRole"], people_df["DayLocationType"]))

# ============================================================
# Plot
# ============================================================

plt.figure(figsize=(8, 5))

location_counts.plot(kind="bar")

plt.xlabel("Daytime Location Type")
plt.ylabel("Number of People")
plt.title("Daytime Location Assignment")

plt.xticks(rotation=30, ha="right")
plt.grid(axis="y")

plt.show()

In [ ]:
# Define Non-Residential Buildings
# Non-residential buildings are used as part of daytime destinations for internal movers and incoming commuters.

non_residential_buildings = buildings_bbox[
    buildings_bbox["BuildingType"].isin(["commercial", "industrial"])
].copy()

non_residential_buildings = non_residential_buildings.reset_index(drop=True)

print("Non-residential buildings:", len(non_residential_buildings))
print(non_residential_buildings["BuildingType"].value_counts())

In [ ]:
# Assign Daytime Building Locations
#The rule is:
# People who remain at home keep their nighttime residential building.
# People assigned to non-residential environments are randomly assigned to one non-residential building.
# People in open spaces are not assigned to a building ID.

nonres_ids = non_residential_buildings["BuildingID"].tolist()

def assign_day_building(row):

    if row["DayLocationType"] == "home":
        return row["HomeBuildingID"]

    elif row["DayLocationType"] == "non_residential":

        if len(nonres_ids) == 0:
            return None

        return random.choice(nonres_ids)

    else:
        return None


people_df["DayBuildingID"] = people_df.apply(assign_day_building, axis=1)

people_df.head()

In [ ]:
# Assign Open-Space Individuals to Random Grid Locations
# Individuals assigned to "open_space" are distributed randomly across the study area.

open_people = people_df[
    people_df["DayLocationType"] == "open_space"
].copy()

# Generate random coordinates inside the bbox
open_people["lon"] = np.random.uniform(
    xmin,
    xmax,
    len(open_people)
)

open_people["lat"] = np.random.uniform(
    ymin,
    ymax,
    len(open_people)
)

# Convert to GeoDataFrame
open_people_gdf = gpd.GeoDataFrame(
    open_people,
    geometry=gpd.points_from_xy(
        open_people["lon"],
        open_people["lat"]
    ),
    crs="EPSG:4326"
)

print("Open-space individuals:", len(open_people_gdf))

open_people_gdf.head()

In [ ]:
# ============================================================
# Final Daytime Population CSV
# ============================================================

final_day_df = people_df.copy()

# Add empty coordinate columns
final_day_df["lon"] = np.nan
final_day_df["lat"] = np.nan

# Fill lon/lat only for open-space people
final_day_df.loc[open_people_gdf.index, "lon"] = open_people_gdf["lon"]
final_day_df.loc[open_people_gdf.index, "lat"] = open_people_gdf["lat"]

# Columns we want, only if they exist
wanted_cols = [
    "PersonID",
    "HouseholdID",
    "HomeBuildingID",
    "DayRole",
    "DayLocationType",
    "DayBuildingID",
    "lon",
    "lat"
]

existing_cols = [c for c in wanted_cols if c in final_day_df.columns]

final_day_df = final_day_df[existing_cols].copy()

# Save
os.makedirs("outputs", exist_ok=True)

final_day_df.to_csv(
    "outputs/final_daytime_population.csv",
    index=False
)

print("Saved: outputs/final_daytime_population.csv")
print("Rows:", len(final_day_df))
print("Columns:", list(final_day_df.columns))